# 00 -- Environment and Provenance

This notebook establishes the three provenance concepts every later notebook in this series relies on, and never re-derives:

1. **Strategy specification provenance** -- `report_current.html`, controls trading logic.
2. **Implementation provenance** -- this repository's own Git commit.
3. **Dataset/result provenance** -- `DataProvenanceManifest`, covered starting in notebook 02.

It also runs the real dependency-availability check every later notebook's "blocked" results trace back to.

In [ ]:
import subprocess
import sys
import atlas_quant

print("python:", sys.version)
print("atlas_quant module path:", atlas_quant.__file__)
try:
    commit = subprocess.run(
        ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True, cwd="../../../..",
    ).stdout.strip()
except Exception as exc:  # pragma: no cover -- git may be unavailable in some environments
    commit = f"<unavailable: {exc}>"
print("git commit:", commit)


## Dependency status

`atlas_quant.dependency_status` never installs anything -- it only reports what is importable right now. The core package (pandas/numpy) must always be available; scikit-learn/requests are only required once a **genuine** production backtest is attempted.

In [ ]:
from atlas_quant.dependency_status import build_environment_report, missing_required_for_production

report = build_environment_report()
for status in report:
    print(f"{status.name:28s} {status.category.value:18s} {status.availability.value:45s} {status.installed_version}")


In [ ]:
missing = missing_required_for_production(report)
print(f"{len(missing)} dependency(ies) required for a genuine production backtest are missing:")
for status in missing:
    print(" -", status.name, "--", status.detail)


## Findings

- The core package imports cleanly without scikit-learn, requests, pyarrow, or jupyter/nbformat installed.
- As of the environment this notebook was authored in, scikit-learn and requests are **not installed**, so any attempt at a genuine production backtest reports `BLOCKED_MISSING_DEPENDENCY` (see notebook 08). This is expected and documented, not a bug. (`hmmlearn` is no longer a dependency at all: the HMM/Markov regime gate was removed from this strategy.)

## Limitations

- This notebook does not install anything (per this project's explicit constraint against installing packages silently). To unblock a genuine run, install the `model`/`production-data` optional dependency groups declared in `pyproject.toml` yourself.